In [1]:
import subprocess
import shutil
import logging
import getpass
import uuid
import os
import sys
usr_name = getpass.getuser()

import warnings
warnings.filterwarnings('ignore')

from IPython.display import clear_output

import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import seaborn as sns
import numpy as np
np.bool = np.bool_
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pandas as pd
from tqdm import tqdm
sys.path.append('/home/21417984_omega-sbrf-ru/notebooks/')
import spark_utils

import json

pd.set_option('display.max_columns', 150)
from datetime import datetime, timedelta
import pickle

In [2]:
import os
def get_spark_session(name, level):
    """
    Get spark context
    :: name - set your app name
    :: level - set max resources level
    """
    python_path = sys.executable
    kernel = python_path.split('/')[-3]
    os.environ['SPARK_MAJOR_VERSION'] = '3'
    os.environ['SPARK_HOME'] = '/usr/sdp/current/spark3-client/'
    os.environ['PYSPARK_DRIVER_PYTHON'] = python_path
    os.environ['PYSPARK_PYTHON'] = python_path
    os.environ['LD_LIBRARY_PATH'] = '/opt/python/virtualenv/jupyter/lib'
    sys.path.insert(0, '/usr/sdp/current/spark3-client/python/')
    sys.path.insert(0, '/usr/sdp/current/spark3-client/python/lib/py4j_current')
 
    # Resources Level Profiles                           #  cpu --  ram -- desc
    if level == 1: lv = ['basic',2,10,2,10,2,2,10]       #   21 --  142 -- для базовых запросов (show create table tbl, show partitions tbl)
    if level == 2: lv = ['basic+CPU',2,10,2,10,2,2,20]   #   41 --  262 -- для простой аналитики (select * from limit 100, sum/count/avg)
    if level == 3: lv = ['middle',4,28,6,28,6,4,20]      #   81 --  742 -- для агрегатов за период 1-2мес (client_aggr_mnth, epk_campaign_daily)
    if level == 4: lv = ['middle+CPU',4,28,6,28,6,4,25]  #  101 --  912 -- для агрегатов за период >1-6мес  (client_aggr_mnth, epk_campaign_daily)
    if level == 5: lv = ['high',4,28,6,36,8,6,30]        #  121 -- 1100 -- для детальных таблиц с большими партициями (_sbol, _card, _eps)
    if level == 6: lv = ['high+CPU',4,18,5,36,8,6,40]    #  161 -- 1000 -- для детальных таблиц с мелкими партициями (feedbacks)
    if level == 7: lv = ['unfriendly',5,28,6,44,10,8,40] #  201 -- 1458 -- для запуска вечером/ночью или на пустом кластере (не рекомендуется)
    lvname = f'{level}.{lv[0]}({lv[1]*lv[7]+1},{lv[4]+lv[2]*lv[7]})'
    print(f'Kernel: {kernel}, Python_path: {python_path}, Resource_level: {lvname}')
    
    # Spark Config      
    from pyspark import SparkContext, SparkConf
    from pyspark.sql import SparkSession
  
    conf = SparkConf().setAppName(f'{name} \n ::{kernel}::{lvname}::')\
        .setMaster("yarn")\
        .set('spark.executor.cores',                     f'{lv[1]}')\
        .set('spark.executor.memory',                    f'{lv[2]}g')\
        .set('spark.executor.memoryOverhead',            f'{lv[3]}g')\
        .set('spark.driver.memory',                      f'{lv[4]}g')\
        .set('spark.driver.memoryOverhead',              f'{lv[5]}g')\
        .set('spark.driver.maxResultSize', '10g')\
        .set('spark.dynamicAllocation.initialExecutors', f'{lv[6]}')\
        .set('spark.dynamicAllocation.maxExecutors',     f'{lv[7]}')\
        .set('spark.dynamicAllocation.enabled', 'true')\
        .set('spark.dynamicAllocation.executorIdleTimeout', '120s')\
        .set('spark.dynamicAllocation.cachedExecutorIdleTimeout', '600s')\
        .set('spark.hive.mapred.supports.subdirectories', 'true')\
        .set('spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive', 'true')\
        .set('spark.shuffle.service.enabled', 'true')\
        .set('spark.port.maxRetries', '150')\
       .set('spark.sql.parquet.writeLegacyFormat', 'true')\
        .set('spark.kerberos.access.hadoopFileSystems','hdfs://arnsdpsbx:8020/')\
        .set('spark.sql.autoBroadcastJoinThreshold','20971520')
    
    spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()
    return spark

try: spark
except NameError: print('Spark3 Starting')
else:
    print('Spark3 Restarting')
    spark.stop()
    
spark = get_spark_session('platon_scoring_mvs', 6) # For example, MyPySpark3
  
import pyspark.sql.functions as F
from pyspark.sql.types import *
import pyspark.sql.types as T
  
sc = spark.sparkContext
sc.setLogLevel('OFF')  # or 'INFO' or 'WARN' or 'OFF'
spark


Spark3 Starting
Kernel: mlpy3811v23, Python_path: /data/sdp/mlpy3811v23/bin/python, Resource_level: 6.high+CPU(161,756)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/24 14:09:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/09/24 14:09:51 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/09/24 14:09:51 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/09/24 14:09:51 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/09/24 14:09:55 WARN HiveConf: HiveConf of name hive.mapred.supports.subdirectories does not exist
25/09/24 14:09:56 WARN Client: Exception encountered while connecting to the server 
org.apache.hadoop.ipc.RemoteException(org.apache.hadoop.ipc.StandbyException): Operation category READ is not supported in state standby. Visit https://s.apache.org/sbnn-error
	at org.apache.hadoop.security.SaslRpcClient.saslConnect(SaslRpcClient.java:376)
	at org.apache.hadoop.ipc.Clien

In [3]:
spark

# PROM

In [4]:
def fix_spark_types(df):
    for col, dtype in df.dtypes:
            if "decimal" in dtype:
                df = df.withColumn(col, F.col(col).cast(T.DoubleType()))

    date_columns = [i for i in df.columns if '_dt' in i]
    for col in date_columns:
        df = df.withColumn(col, F.when(F.col(col) > F.to_date(F.lit(pd.Timestamp.max)), F.to_date(F.lit(pd.Timestamp.max))).otherwise(F.col(col)))
    
    return df

In [5]:
with open ('features_dict.pkl', 'rb') as f:
    feature_names = pickle.load(f)

In [6]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

def get_diff_dates(report_dt: str) -> tuple:
    
    current_date = datetime.strptime(report_dt, '%Y-%m-%d')
    
    date_3m = current_date - relativedelta(months = +2)
    next_mnth_3 = date_3m.replace(day=28) + timedelta(days = 4)
    date_3m = datetime.strftime(next_mnth_3 - timedelta(days=next_mnth_3.day), '%Y-%m-%d')
    date_3m_1 = datetime.strftime(datetime.strptime(date_3m, '%Y-%m-%d').replace(day = 1), '%Y-%m-%d')
    
    

    date_6m = current_date - relativedelta(months = +5)
    next_mnth_6 = date_6m.replace(day=28) + timedelta(days = 4)
    date_6m = datetime.strftime(next_mnth_6 - timedelta(days=next_mnth_6.day), '%Y-%m-%d')
    date_6m_1 = datetime.strftime(datetime.strptime(date_6m, '%Y-%m-%d').replace(day = 1), '%Y-%m-%d')
    
    
    
    date_12m = current_date - relativedelta(months = +11)
    next_mnth_12 = date_12m.replace(day=28) + timedelta(days = 4)
    date_12m = datetime.strftime(next_mnth_12 - timedelta(days=next_mnth_12.day), '%Y-%m-%d')
    date_12m_1 = datetime.strftime(datetime.strptime(date_12m, '%Y-%m-%d').replace(day = 1), '%Y-%m-%d')
    
    return date_3m, date_3m_1, date_6m, date_6m_1, date_12m, date_12m_1

In [7]:
report_dt = '2025-07-31'
date_now = report_dt
date_3m, date_3m_1, date_6m, date_6m_1, date_12m, date_12m_1 = get_diff_dates(report_dt)
report_dt_1 = datetime.strftime(datetime.strptime(report_dt, '%Y-%m-%d').replace(day = 1), '%Y-%m-%d')

date_now_name = date_now.replace('-', '_')

In [8]:
date_12m

'2024-08-31'

In [9]:
from catboost import CatBoostClassifier

In [10]:
model = CatBoostClassifier()
model.load_model('../models/final_model.cbm')

In [11]:
model.feature_names_

['tp_active_kind_cd',
 'sd_age_yrs_frac_nv',
 'tot_bal_with_invest_12m',
 'tp_mnth_lst_grace_end_exp_qty',
 'tp_1st_open_dt',
 'crd_otf_total_rub_amt_12m',
 'insur_total_bal',
 'tp_best_kind_ever_cd',
 'dep_acct_dep_mnth_lst_open_qty',
 'full_torg_pos_pos_12m',
 'seg_client_fl_segment_cd',
 'srv_sbol_mnth_web_lst_log_qty',
 'seg_age_segment',
 'prl_client_app_income_amt',
 'dep_acct_dep_td_bal_rub_amt',
 'tp_mnth_lst_close_qty',
 'tot_bal_with_invest_3m',
 'dep_topup_12m_avg_rub_amt',
 'lne_lst_open_pl_rate',
 'srv_sbol_mnth_1st_txn_qty',
 'dep_acct_dep_save_bal_rub_amt',
 'crd_dc_mnth_snc_open_qty',
 'pfm_opened_cnt_all_12m',
 'prd_lst_prod_tb_cd',
 'dep_acct_tot_bal_prev_rub_amt',
 'srv_ap_othr_1st_txn_ever_dt',
 'dep_acct_dep_td_qty',
 'lne_pl_clsd_wavg_intr_rate']

In [12]:
ft_client_aggr_mnth_table = 'prx_bpm_client_aggr_custom_rozn_client_aggr.ft_client_aggr_mnth'
feedback_mon_card = 'prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_card_transactions'
feedback_mon_feedback = 'prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_feedbacks'

In [13]:
agg = spark.sql(f'''
select distinct epk_id, report_dt, tp_active_kind_cd, sd_age_yrs_frac_nv, tp_1st_open_dt, insur_total_bal,
tp_best_kind_ever_cd, seg_client_fl_segment_cd, seg_age_segment, prl_client_app_income_amt,
dep_acct_dep_td_bal_rub_amt, dep_topup_12m_avg_rub_amt, lne_lst_open_pl_rate, dep_acct_dep_save_bal_rub_amt,
prd_lst_prod_tb_cd, dep_acct_tot_bal_prev_rub_amt, srv_ap_othr_1st_txn_ever_dt, lne_pl_clsd_wavg_intr_rate,
tp_mnth_lst_grace_end_exp_qty, dep_acct_dep_mnth_lst_open_qty, srv_sbol_mnth_web_lst_log_qty, 
tp_mnth_lst_close_qty, srv_sbol_mnth_1st_txn_qty, crd_dc_mnth_snc_open_qty, dep_acct_dep_td_qty
from {ft_client_aggr_mnth_table}
where report_dt = '{date_now}' and cla_full_active_nflag = 1 and sd_dead_nflag = 0 and seg_client_cx_segment_cd = 'MVS'
''')

In [14]:
feedback = spark.sql(f'''
with main_epk as (select distinct epk_id, report_dt from {ft_client_aggr_mnth_table}
where report_dt = '{date_now}' and cla_full_active_nflag = 1 and sd_dead_nflag = 0 and seg_client_cx_segment_cd = 'MVS'),

feedback_1m as (select epk_id, full_torg_pos_pos_12m, report_dt from {feedback_mon_card}
where report_dt = '{date_now}'),

feedback_12m as (select epk_id, sum(opened_feedbacks_cnt) as pfm_opened_cnt_all_12m from {feedback_mon_feedback}
where report_dt between '{date_12m}' and '{date_now}'
group by epk_id)

select distinct epk_id, main_epk.report_dt, full_torg_pos_pos_12m, pfm_opened_cnt_all_12m
from main_epk
left join feedback_1m using(epk_id)
left join feedback_12m using(epk_id)
''')

In [15]:
flows_deep = spark.sql(f'''
with target as (select distinct epk_id, report_dt
from {ft_client_aggr_mnth_table}
where report_dt = '{date_now}' and cla_full_active_nflag = 1 and sd_dead_nflag = 0 and seg_client_cx_segment_cd = 'MVS'
),

flows as (select epk_id, report_dt, (coalesce(dep_tot_bal_rub_amt, 0) +
                            coalesce(inv_mf_agrmnt_bal_rub_amt, 0) +
                            coalesce(inv_tm_agrmnt_bal_rub_amt, 0) +
                            coalesce(inv_bo_agrmnt_bal_tot_rub_amt, 0) +
                            coalesce(bal_invest_insur_life_amt, 0) +
                            coalesce(bal_nakop_insur_life_amt, 0)
                            ) as tot_bal_with_invest,
                        crd_otf_total_rub_amt
FROM
                {ft_client_aggr_mnth_table}
            WHERE
                report_dt BETWEEN '{date_12m}' and '{date_now}'
                        
),

target_deep as (
            SELECT
                epk_id,
                report_dt,
                last_day(add_months(report_dt, -2)) AS report_dt_3m,
                last_day(add_months(report_dt, -11)) AS report_dt_12m
            FROM 
                target   
)

SELECT 
    target_deep.epk_id,
    target_deep.report_dt,
    avg(agg_12m.crd_otf_total_rub_amt) as crd_otf_total_rub_amt_12m, -- Σ списаний по всем картам
    avg(agg_12m.tot_bal_with_invest) as tot_bal_with_invest_12m,
    avg(agg_3m.tot_bal_with_invest) as tot_bal_with_invest_3m
from 
    target_deep
    LEFT JOIN flows agg_3m USING (epk_id)
    LEFT JOIN flows agg_12m USING (epk_id)
where 
    agg_3m.report_dt BETWEEN target_deep.report_dt_3m AND target_deep.report_dt
    AND agg_12m.report_dt BETWEEN target_deep.report_dt_12m AND target_deep.report_dt
GROUP BY
    target_deep.report_dt,
    target_deep.epk_id
''')

In [17]:
df = agg.join(feedback, how = 'left', on = ['epk_id', 'report_dt'])
df = df.join(flows_deep, how = 'left', on = ['epk_id', 'report_dt'])

In [30]:
df_fix = fix_spark_types(df)
df_fix.write.saveAsTable(f'arnsdpsbx_team_ss.bpm_scoring_multiclass_premier_features_{date_now_name}', mode='overwrite')

In [31]:
import logging
logging.basicConfig(level=logging.INFO, format='[%(asctime)s %(levelname)s - %(message)s]')

def normalize_column_names(data):
    logging.info("Приведение колонок к нижнему регистру")
    rename_map = {col: col.lower() for col in data.columns if col != col.lower()}
    return data.rename(columns=rename_map)


def process_date_columns(data, report_col):
    logging.info("Преобразование datetime колонок в 'дни до report_dt'")
    if report_col in data.columns:
        data[report_col] = pd.to_datetime(data[report_col].astype(str), errors="coerce")
    date_cols = [col for col in data.columns if "_dt" in col != report_col]
    for col in tqdm(date_cols, desc="Обработка дат"):
        data[col] = pd.to_datetime(data[col].astype(str), errors="coerce")
        data[col] = (data[report_col] - data[col]).dt.days
    return data.drop(columns=[report_col], errors="ignore")
    # return data

def fill_missing_values(data):
    EXCLUDED_COLS = ["report_dt"]
    logging.info("Заполнение пропусков")
    fill_config = {
        "flg":               {"fill": -1,   "type": "int32"},
        "_qty":              {"fill": -999999},
        "_pct":              {"fill": -999999},
        "_days":             {"fill": -999999},
        "_num":              {"fill": -999999},
        "_amt":              {"fill": -999999},
        "_prc":              {"fill": -999999},
        "_rate":             {"fill": -999999,   "type": "float64"},
        "_frac":             {"fill": -999999,   "type": "float64"},
        "_nflag":            {"fill": -999999,   "type": "float64"},
        "_dt":               {"fill": -999999,   "type": "float64"},
        "float64":           {"fill": -999999},
        "int32":             {"fill": -999999},
        "object":            {"fill": 'UNKNOWN'}
    }
    
    for col in tqdm(data.columns, desc="Заполнение пропусков"):
        if col in EXCLUDED_COLS:
            continue
        col_handled = False
        
        for pattern, rule in fill_config.items():
            if pattern in col:
                fill_val = rule["fill"]
                dtype = rule.get("type", None)
                data[col] = data[col].fillna(fill_val)
                if dtype:
                    data[col] = data[col].astype(dtype)
                col_handled = True
                break
                
        if not col_handled:
            col_type = str(data[col].dtype)
            if col_type in fill_config:
                fill_val = fill_config[col_type]["fill"]
                dtype = fill_config[col_type].get("type", None)
                data[col] = data[col].fillna(fill_val)
                if dtype:
                    data[col] = data[col].astype(dtype)
                    
    return data

def preprocess(data):
    logging.info("Старт предобработки")
    data = normalize_column_names(data)
    data = process_date_columns(data, report_col="report_dt")
    data = fill_missing_values(data)
    cat_features = data.select_dtypes(include="object").columns.tolist()
    logging.info(f"предобработка завершена. Число признаков: {data.shape[1]-1}")
    return data, cat_features

In [32]:
from catboost import CatBoostClassifier

model = CatBoostClassifier()
model.load_model('../models/final_model.cbm')
name_feature = model.feature_names_
model = spark.sparkContext.broadcast(model)

In [33]:
def func_(it):
    for pdf in it:
        clf = model.value
        f = clf.feature_names_
        prep = preprocess(pdf)
        x = prep[0] if isinstance(prep, tuple) else prep
        x = x[f]
        proba = clf.predict_proba(x)
        labels = getattr(clf, 'classes_', None)
        if labels is None:
            labels = list(range(1, proba.shape[1] + 1))
        labels = [int(x) for x in labels]
        n = len(pdf)
        
        out = pd.DataFrame({
            'epk_id': pdf['epk_id'].astype('int64').values,
            'report_dt': pdf['report_dt'].astype(str).values,
            'cluster_1': np.full(n, labels[0], dtype='int32'),
            'cluster_1_score': proba[:, 0].astype('float64'),
            'cluster_2': np.full(n, labels[1], dtype='int32'),
            'cluster_2_score': proba[:, 1].astype('float64'),
            'cluster_3': np.full(n, labels[2], dtype='int32'),
            'cluster_3_score': proba[:, 2].astype('float64'),
            'cluster_4': np.full(n, labels[3], dtype='int32'),
            'cluster_4_score': proba[:, 3].astype('float64'),
        })
        
        for c in ['cluster_1', 'cluster_2', 'cluster_3', 'cluster_4']:
            out[c] = out[c].astype('int32')
        
        yield out[['epk_id', 'cluster_1', 'cluster_1_score', 
                 'cluster_2', 'cluster_2_score',
                  'cluster_3', 'cluster_3_score',
                  'cluster_4', 'cluster_4_score',
                  'report_dt'
                 ]]

In [34]:
spark_df = spark.read.table(f'arnsdpsbx_team_ss.bpm_scoring_multiclass_premier_features_{date_now_name}')

from pyspark.sql.functions import *
import pyspark.sql.types as T
out_schema = StructType(
    fields = [
        T.StructField('epk_id', T.LongType(), True),
        #T.StructField('cluster_1', T.IntegerType(), True),
        T.StructField('cluster_1_score', T.FloatType(), True),
        #T.StructField('cluster_2', T.IntegerType(), True),
        T.StructField('cluster_2_score', T.FloatType(), True),
        #T.StructField('cluster_3', T.IntegerType(), True),
        T.StructField('cluster_3_score', T.FloatType(), True),
        #T.StructField('cluster_4', T.IntegerType(), True),
        T.StructField('cluster_4_score', T.FloatType(), True),
        T.StructField('report_dt', T.StringType(), True)])

result = spark_df.mapInPandas(func_, out_schema)

schema = 'arnsdpsbx_team_ss'
table_name = f'bpm_scoring_multiclass_premier_{date_now_name}'

result.write.saveAsTable(f'{schema}.{table_name}'
                         , mode='overwrite'
                         , format='parquet')